# AfriVoices — KenLM v3 "costaud" (cible kln/mas, session CPU)

Les LM v2 de **kln/mas** n'ont AUCUN texte externe (pas de Wikipedia). Pour les renforcer
sans corpus externe, trois leviers :
1. **Ordre 6** (au lieu de 5) — capture des séquences de mots plus longues.
2. **Élagage nul** (`--prune 0 0 0`) sur les petites langues — on ne jette aucun n-gram
   (corpus petit -> .bin reste raisonnable).
3. **TOUTES les transcriptions Anv-ke** (train complet + dev), pas seulement celles vues
   à l'entraînement — c'est le vrai gain : du texte natif jamais exploité par le LM.

Sortie dans **lm_v3/** (lm_v2 reste intact). Comparaison finale par grille alpha/beta
sur le dev : on ne garde v3 par langue QUE s'il bat v2. Sinon on recopie v2.

**Session CPU.** Lit les transcriptions depuis le Drive (anvke/), pas de GPU.

## 1. Drive + alphabet + collecte de TOUTES les transcriptions Anv-ke

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, os, re, glob

BASE="/content/drive/MyDrive/afrivoices"
LM_V2=f"{BASE}/lm_v2"
LM_V3=f"{BASE}/lm_v3"
os.makedirs(LM_V3, exist_ok=True)

vocab=json.load(open(f"{BASE}/baobab-asr-v4/vocab.json", encoding="utf-8"))
ALPHABET=set(t for t in vocab.keys() if len(t)==1); ALPHABET.add(" ")

def clean_ext(t):
    t=(t or "").lower()
    t="".join(ch if ch in ALPHABET else " " for ch in t)
    return re.sub(r"\s+"," ",t).strip()

# mapping langue -> dossier Drive
ANVKE=f"{BASE}/anvke"
ALIAS={"kik":["kik","kikuyu","gikuyu"],"luo":["luo","dholuo"],"kln":["kln","kalenjin"],
       "mas":["mas","maasai"],"som":["som","somali"]}
lang_dirs={}
for d in os.listdir(ANVKE):
    dl=d.lower()
    for lang,als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs: lang_dirs[lang]=os.path.join(ANVKE,d)
print("langues:", list(lang_dirs))

In [ ]:
import pandas as pd
# collecte TOUTES les transcriptions (train + dev), lecture colonne texte uniquement.
# ⚠️ lit beaucoup de parquets depuis le Drive -> peut prendre 10-30 min. Barre de suivi.
from tqdm.auto import tqdm
trans_all={}
for lang, root in lang_dirs.items():
    parqs=sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    seen=set()
    for pq in tqdm(parqs, desc=f"{lang}", unit="pq"):
        try:
            df=pd.read_parquet(pq, columns=["transcription"])   # ne lit QUE le texte
        except Exception:
            continue
        for t in df["transcription"].astype(str):
            c=clean_ext(t)
            if len(c.split())>=1: seen.add(c)
    trans_all[lang]=sorted(seen)
    print(f"{lang}: {len(trans_all[lang])} transcriptions uniques")

# comparaison au corpus v2 existant (pour voir le gain de texte natif)
import os
for lang in ["kln","mas"]:
    v2n=len([l for l in open(f"{LM_V2}/{lang}.txt",encoding="utf-8") if l.strip()])
    print(f"  {lang}: v2={v2n} lignes -> v3={len(trans_all[lang])} uniques "
          f"({'+' if len(trans_all[lang])>v2n else ''}{len(trans_all[lang])-v2n})")

## 2. Construire les corpus v3

Pour **kln/mas** : toutes les transcriptions collectées (×1 — elles sont déjà toutes là,
pas besoin de dupliquer un corpus déjà natif).
Pour les autres : on **réutilise le corpus v2** tel quel (déjà enrichi Wikipedia, ne pas
gâcher). On ne "renforce" que kln/mas ici.

In [ ]:
CIBLES=["kln","mas"]     # les langues à renforcer
for lang in ["swa","kik","luo","kln","mas","som"]:
    dst=f"{LM_V3}/{lang}.txt"
    if lang in CIBLES:
        corpus=trans_all[lang]
        open(dst,"w",encoding="utf-8").write("\n".join(corpus))
        print(f"{lang} (RENFORCÉ): {len(corpus)} lignes")
    else:
        # recopier le corpus v2 (déjà optimal)
        src=f"{LM_V2}/{lang}.txt"
        open(dst,"w",encoding="utf-8").write(open(src,encoding="utf-8").read())
        print(f"{lang} (copie v2): inchangé")

## 3. Compiler KenLM : ordre 6 + élagage nul pour kln/mas

In [ ]:
!apt-get install -y build-essential cmake libboost-all-dev libeigen3-dev >/dev/null 2>&1
!git clone https://github.com/kpu/kenlm.git /content/kenlm 2>/dev/null
!cd /content/kenlm && mkdir -p build && cd build && cmake .. >/dev/null 2>&1 && make -j4 >/dev/null 2>&1
!ls /content/kenlm/build/bin/ | head -3

In [ ]:
import os
BIN="/content/kenlm/build/bin"
CIBLES=["kln","mas"]
for lang in ["swa","kik","luo","kln","mas","som"]:
    txt, arpa, binf = f"{LM_V3}/{lang}.txt", f"{LM_V3}/{lang}.arpa", f"{LM_V3}/{lang}.bin"
    if lang in CIBLES:
        order, prune = 6, "0 0 0"      # ordre 6, aucun élagage (petit corpus)
    else:
        order, prune = 5, "0 0 1"      # comme v2
    os.system(f"{BIN}/lmplz -o {order} --discount_fallback --prune {prune} <{txt}> {arpa} 2>/tmp/{lang}.log")
    os.system(f"{BIN}/build_binary {arpa} {binf} 2>>/tmp/{lang}.log")
    ok=os.path.exists(binf) and os.path.getsize(binf)>0
    sz=os.path.getsize(binf)/1e6 if ok else 0
    print(f"{lang}: {'OK' if ok else 'ECHEC voir /tmp/'+lang+'.log'}  ordre={order} prune=({prune})  bin={sz:.0f} Mo")
    if os.path.exists(arpa): os.remove(arpa)

## 4. Comparaison v2 vs v3 sur le dev (garder le meilleur par langue)

⚠️ Nécessite une session GPU (transcrire le dev). Alternative : fais cette comparaison
dans ton notebook de décodage habituel en pointant LM sur lm_v3 vs lm_v2, grille (0.7,0.5).
Ci-dessous, version autonome minimale si le modèle est accessible.

In [ ]:
# --- à lancer en session GPU (ou copier la logique dans le notebook de grille) ---
# Compare, pour kln et mas seulement, le WER dev avec LM v2 vs LM v3.
# Si v3 < v2 pour une langue -> garder v3 ; sinon recopier v2 dans lm_v3.
print("Compare lm_v2 vs lm_v3 pour kln/mas dans ton notebook de décodage :")
print("  - construit 2 décodeurs par langue (un par LM), alpha=0.7 beta=0.5")
print("  - garde le LM qui donne le WER dev le plus bas")
print("Si lm_v3 ne bat pas lm_v2 pour une langue, copie simplement lm_v2/{lang}.* -> lm_v3/")
print("\nCommande de secours pour recopier v2 si v3 déçoit sur une langue :")
print('  import shutil; [shutil.copy(f"{LM_V2}/kln."+e, f"{LM_V3}/kln."+e) for e in ["bin","txt"]]')

## 5. Décision

- Si **lm_v3 bat lm_v2** sur kln/mas au dev → utilise lm_v3 pour ces langues à la
  soumission (les autres langues sont identiques, lm_v3 les a recopiées de lm_v2).
- Sinon → aucun gain de ce levier, reste sur lm_v2. C'est une info utile : ça confirme
  que le plateau kln/mas est acoustique, pas linguistique.

⚠️ Re-mesure la taille du **bin le plus gros** (swa recopié = 819 Mo) pour le rapport edge :
inchangé côté swa, kln/mas ordre 6 restent petits (< 200 Mo). RAM edge OK.